[![Mở trong Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trungnguyenvn/mlsysbook-vi-notebooks/blob/main/vol1-ch09-data-selection/03-chon-loc-dong.ipynb)

# Chọn lọc động: học theo lộ trình, học chủ động, học bán giám sát

Cắt tỉa tĩnh chọn một lần, trước khi huấn luyện. Chọn lọc động quyết định **trong lúc** huấn luyện: đưa
mẫu nào vào trước, hỏi nhãn cho mẫu nào, và tin dự đoán nào của chính mô hình. Sổ tay này đo ba kỹ thuật
ấy trên FashionMNIST và MNIST, mỗi kỹ thuật so với một mốc chọn ngẫu nhiên.

**Bạn sẽ làm:**
1. tính lại bảng 5 và hàm nhịp độ tuyến tính, rồi đo học theo lộ trình (curriculum learning) so với xáo
   trộn ngẫu nhiên, ở cùng số mẫu đã xử lý;
2. chạy một vòng học chủ động (active learning) với bốn chiến lược truy vấn, đếm số nhãn mỗi chiến lược
   cần để đạt một mục tiêu, và xem truy vấn dồn vào đâu;
3. tính lại phép tính nhẩm 1.3 của bệnh viện, rồi cân cỡ lô truy vấn với độ trễ gán nhãn;
4. học bán giám sát (semi-supervised learning) với 500 nhãn trên 50,000 ảnh: gán nhãn giả theo ngưỡng,
   ảnh ngoài phân phối lọt vào kho, và lan truyền nhãn;
5. tính lại bảng 6 của FixMatch.

In [ ]:
#@title Chuẩn bị: thư viện, hạt giống và hai hàm đối chiếu với sách { display-mode: "form" }
# Chạy ô này trước. Nó không cài thêm gì: mọi thư viện dùng ở đây đều có sẵn trên Colab.
import math, os, random, re, sys, time

import matplotlib.pyplot as plt
import numpy as np
import torch

# Số luồng CPU. 0 để thư viện tự chọn (trên Colab là 2).
NB_THREADS = int(os.environ.get("NB_THREADS", "0"))
if NB_THREADS:
    torch.set_num_threads(NB_THREADS)
    try:
        torch.set_num_interop_threads(NB_THREADS)
    except RuntimeError:  # chỉ đặt được một lần mỗi phiên
        pass
    from threadpoolctl import threadpool_limits
    threadpool_limits(NB_THREADS)

SEED = 42


def dat_hat_giong(seed=SEED):
    """Cùng hạt giống thì cùng kết quả, mỗi lần chạy lại."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


dat_hat_giong()
DATA = os.environ.get("NB_DATA", "data")  # nơi tải dữ liệu về
NB_STRICT = os.environ.get("NB_STRICT") == "1"
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.3})

# Đọc một con số theo cách bảng số liệu của bài học viết nó (dấu chấm hay dấu phẩy).
import math
import re

_SUP = str.maketrans("⁰¹²³⁴⁵⁶⁷⁸⁹⁻⁺", "0123456789-+")
_MULT = {"tỷ": 1e9, "tỉ": 1e9, "triệu": 1e6, "nghìn": 1e3, "ngàn": 1e3,
         "K": 1e3, "M": 1e6, "B": 1e9}
_NUM = re.compile(
    r"(?<![\w.,])(-?)(\d+(?:[.,]\d+)*)"                      # sign, digits with separators
    r"(?:\s*[×x·]\s*10\s*(?:\^\{?\s*([-+]?\d+)\s*\}?|([⁻⁺]?[⁰¹²³⁴⁵⁶⁷⁸⁹]+))|[eE]([-+]?\d+)"
    r"|\^\{?\s*([-+]?\d+)\s*\}?|([⁻⁺]?[⁰¹²³⁴⁵⁶⁷⁸⁹]+))?"                  # or a bare power, 10^{-5}
    r"(\s*%|\s*(?:tỷ|tỉ|triệu|nghìn|ngàn)(?!\w)|\s?[KMB](?![A-Za-z]))?")


def norm(text):
    """The outline's markup and typography reduced to plain text, as both sides compare it."""
    text = text.replace("{,}", ",").replace("**", "").replace("`", "")
    text = re.sub(r"[   ]", " ", text)
    text = re.sub(r"−(?=\d)", "-", text)
    text = re.sub(r"[~≈]", "", text)
    return re.sub(r"\s+", " ", text).strip()


def _readings(digits):
    """(value, decimals) for each way the separators in `digits` can be read."""
    out = []
    groups_ok = lambda parts: all(len(p) == 3 for p in parts[1:])
    for thou, dec in ((",", "."), (".", ",")):
        if thou in digits and dec in digits:
            head, _, tail = digits.rpartition(dec)
            if dec in head or not tail.isdigit() or not groups_ok(head.split(thou)):
                continue
            out.append((float(head.replace(thou, "") + "." + tail), len(tail)))
    for sep in (",", "."):
        other = "." if sep == "," else ","
        if sep in digits and other not in digits:
            parts = digits.split(sep)
            if groups_ok(parts):
                out.append((float("".join(parts)), 0))                    # thousands
            if len(parts) == 2:
                out.append((float(parts[0] + "." + parts[1]), len(parts[1])))  # decimal
    if "." not in digits and "," not in digits:
        out.append((float(digits), 0))
    return out


def candidates(text):
    """Every (value, half_unit) the numbers in `text` can mean.

    half_unit is half of the last printed digit, scaled like the value, so `437,5 KB`
    accepts 437.544. Each number is offered as written and, when a multiplier or `%`
    follows it, scaled by that too, so `8,5 tỉ` cites 8.5 or 8.5e9 and `20 %` cites 20 or 0.2.
    """
    out = []
    for m in _NUM.finditer(norm(text)):
        sign, digits, exp_caret, exp_sup, exp_e, pow_caret, pow_sup, suffix = m.groups()
        exp = exp_caret or (exp_sup.translate(_SUP) if exp_sup else None) or exp_e
        power = pow_caret or (pow_sup.translate(_SUP) if pow_sup else None)
        scale10 = 10.0 ** int(exp) if exp else 1.0
        suffix = (suffix or "").strip()
        extra = 0.01 if suffix == "%" else _MULT.get(suffix)
        for value, decimals in _readings(digits):
            if power:
                # 10^{-5}, 2³²: an exact value, so it is matched to 1 part in 10⁹; a looser
                # comparison needs an explicit tol.
                v = value ** int(power)
                half = abs(v) * 1e-9
            else:
                v = value * scale10
                half = 0.5 * 10.0 ** -decimals * scale10
            v = -v if sign else v
            out.append((v, half))
            if extra:
                out.append((v * extra, half * extra))
    return out


def match(value, text):
    """The (value, half_unit) reading of `text` that `value` equals, or None."""
    for v, half in candidates(text):
        if math.isclose(value, v, rel_tol=1e-9, abs_tol=1e-300):
            return v, half
    return None


def well_anchored(text):
    """False for a citation too bare to identify one cell, like "1" or "2"."""
    t = norm(text)
    rest = re.sub(r"[\d.,\s\-+]", "", t)
    if rest:
        return True
    sig = re.sub(r"[^\d]", "", t).lstrip("0")
    return len(sig) >= 2


def _so(x):
    """Một con số để in: số nguyên có dấu phẩy ngăn nghìn, số thực giữ 6 chữ số có nghĩa."""
    if isinstance(x, (int, np.integer)) or (isinstance(x, float) and x.is_integer() and abs(x) < 1e15):
        return f"{int(x):,}"
    return f"{x:,.6g}"


def sach(gia_tri, nguon=None, trich=None, tol=None):
    """Một con số của sách, kèm đúng chữ của nó: trong bảng số liệu của bài học (nguon)
    hoặc nguyên văn trong chương (trich)."""
    chu = nguon if nguon is not None else trich
    if match(gia_tri, chu) is None:
        raise ValueError(f"{gia_tri!r} không phải con số ghi trong {chu!r}")
    return gia_tri


def theo_sach(ten, gia_tri, sach, nguon=None, trich=None, tol=None):
    """So một con số tính được với con số của sách. In ✓ khi khớp, ✗ kèm độ lệch khi không."""
    chu = nguon if nguon is not None else trich
    khop = match(sach, chu)
    if khop is None:
        raise ValueError(f"{sach!r} không phải con số ghi trong {chu!r}")
    sai_so = tol if tol is not None else khop[1]
    dung = abs(gia_tri - sach) <= sai_so
    print(f"{'✓' if dung else '✗'} {ten}: tính được {_so(gia_tri)} · sách: {chu}")
    if not dung:
        loi = f"lệch {_so(gia_tri - sach)}, quá sai số cho phép {_so(sai_so)}"
        if NB_STRICT:
            raise AssertionError(f"{ten}: {loi}")
        print(f"   {loi}. Nếu bạn vừa đổi tham số ở trên thì đây là điều được chờ đợi.")
    # None, not the bool: a cell ending in theo_sach(...) would otherwise echo True under the ✓.


def do_tren_may(ten, gia_tri, don_vi=""):
    """Một con số ĐO trên máy đang chạy: mỗi máy một khác, không phải con số của sách."""
    print(f"⏱ {ten}: {_so(gia_tri)} {don_vi} (đo trên máy này)".replace("  ", " "))


print(f"Python {sys.version.split()[0]} · PyTorch {torch.__version__} · NumPy {np.__version__}"
      f" · {torch.get_num_threads()} luồng CPU")

In [ ]:
import warnings

import torch.nn.functional as F
import torchvision
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.semi_supervised import LabelSpreading

# Hồi quy logistic dừng ở 200 vòng lặp, đủ cho mọi phép so ở đây; cảnh báo chưa hội tụ hẳn được tắt.
warnings.filterwarnings("ignore", category=ConvergenceWarning)

thoi_trang = torchvision.datasets.FashionMNIST(DATA, train=True, download=True)
thoi_trang_kt = torchvision.datasets.FashionMNIST(DATA, train=False, download=True)
N = 10_000
X = thoi_trang.data[:N].reshape(-1, 784).float() / 255
y = thoi_trang.targets[:N].clone()
X_kt = thoi_trang_kt.data[:5_000].reshape(-1, 784).float() / 255
y_kt = thoi_trang_kt.targets[:5_000].clone()

chu_so = torchvision.datasets.MNIST(DATA, train=True, download=True)
chu_so_kt = torchvision.datasets.MNIST(DATA, train=False, download=True)
A = chu_so.data.reshape(-1, 784).numpy().astype(np.float32) / 255
nhan = chu_so.targets.numpy()
A_kt = chu_so_kt.data.reshape(-1, 784).numpy().astype(np.float32) / 255
nhan_kt = chu_so_kt.targets.numpy()

# 50 thành phần chính, tính trên 20,000 ảnh chữ số chưa dùng nhãn, làm đặc trưng cho mọi mô hình logistic.
tam = A[:20_000].mean(0)
_, _, Vt = np.linalg.svd(A[:20_000] - tam, full_matrices=False)
chieu = Vt[:50].T
do_lech = ((A - tam) @ chieu).std(0)


def dac_trung(a):
    return (a - tam) @ chieu / do_lech


Z, Z_kt = dac_trung(A), dac_trung(A_kt)
print(f"FashionMNIST: {N:,} ảnh huấn luyện, {len(X_kt):,} ảnh kiểm tra")
print(f"MNIST: {len(A):,} ảnh huấn luyện, {len(A_kt):,} ảnh kiểm tra, mỗi ảnh còn {Z.shape[1]} đặc trưng")

## 1. Học theo lộ trình

Một lộ trình cần hai thành phần: một bộ chấm độ khó để xếp hạng mẫu, và một **hàm nhịp độ** quyết định
mẫu khó vào nhanh đến đâu. Với nhịp độ tuyến tính, ở epoch $n$ mô hình chỉ học trên phần dễ nhất của tập
đã xếp theo độ khó, cỡ $D \cdot \min(1, n / N_{\text{khởi động}})$, rồi học trên cả tập.

Bảng 5 dùng số epoch **giả định** để minh hoạ cách tính phần tiết kiệm; chương nói rõ đây không phải số
đo từ các nghiên cứu được trích. Trước hết, kiểm phép tính.

In [ ]:
def it_hon(lo_trinh, goc):
    """Phần epoch tiết kiệm, theo quy tắc của bảng 5."""
    return 100 * (1 - lo_trinh / goc)


theo_sach("CIFAR-10, ResNet-18, khởi động dần tuyến tính: ít epoch hơn (%)",
          it_hon(sach(115, nguon="115 so với 150"), sach(150, nguon="115 so với 150")),
          sach=23.3, nguon="115 so với 150 · 23.3 phần trăm")
theo_sach("CIFAR-100, ResNet-32, tự điều nhịp: ít epoch hơn (%)",
          it_hon(sach(180, nguon="180 so với 220"), sach(220, nguon="180 so với 220")),
          sach=18.2, nguon="180 so với 220 · 18.2 phần trăm")
theo_sach("ImageNet, ResNet-50, theo mất mát: ít epoch hơn (%)",
          it_hon(sach(80, nguon="80 so với 90"), sach(90, nguon="80 so với 90")),
          sach=11.1, nguon="80 so với 90 · 11.1 phần trăm")
theo_sach("ImageNet, ResNet-50, học được (nhãn nhiễu): ít epoch hơn (%)",
          it_hon(sach(70, nguon="70 so với 90"), sach(90, nguon="70 so với 90")),
          sach=22.2, nguon="70 so với 90 · 22.2 phần trăm")

# Hàm nhịp độ tuyến tính: phần của tập được dùng ở mỗi epoch.
SO_EPOCH, KHOI_DONG = 12, 6
phan_dung = [min(1.0, n / KHOI_DONG) for n in range(1, SO_EPOCH + 1)]
print(f"\n{SO_EPOCH} epoch, khởi động dần {KHOI_DONG} epoch: tổng cộng {sum(phan_dung):.1f} epoch-tương-đương,"
      f" tức {sum(phan_dung) / SO_EPOCH:.1%} số mẫu của lịch xáo trộn ngẫu nhiên")

fig, ax = plt.subplots(figsize=(7, 2.8))
ax.bar(range(1, SO_EPOCH + 1), phan_dung, color="C2", alpha=0.8, label="lộ trình, nhịp độ tuyến tính")
ax.step(np.arange(0.5, SO_EPOCH + 1), [1] * (SO_EPOCH + 1), where="post", color="0.3", lw=1.2, label="xáo trộn ngẫu nhiên")
ax.set_xticks(range(1, SO_EPOCH + 1))
ax.set_xlabel("epoch")
ax.set_ylabel("phần của tập được dùng")
ax.set_ylim(0, 1.45)
ax.set_title(f"Hàm nhịp độ tuyến tính, khởi động dần {KHOI_DONG} epoch")
ax.legend(loc="upper left", fontsize=8, ncol=2)
plt.tight_layout()
plt.show()

Diện tích cột xanh là số mẫu lộ trình xử lý: trong giai đoạn khởi động dần nó học ít mẫu hơn, nên ở cùng
số epoch nó tốn ít tính toán hơn. Câu hỏi là độ chính xác có theo kịp không. Bảng 4 của chương liệt kê bốn
bộ chấm độ khó; sổ tay thử hai: **theo mất mát** của một mô hình thăm dò nhỏ học 3 epoch, và **tự điều
nhịp**, dùng chính mất mát hiện tại của mô hình, cập nhật mỗi epoch. Thêm hai biến thể: lộ trình chia đều
theo lớp, và lộ trình ngược, đưa mẫu khó lên trước.

**Dự đoán:** ở cùng số mẫu đã xử lý, thứ tự nào đạt độ chính xác cao nhất?

In [ ]:
def mlp(an=128):
    return torch.nn.Sequential(torch.nn.Linear(784, an), torch.nn.ReLU(), torch.nn.Linear(an, 10))


def mot_epoch(m, toi_uu, chon, batch=128):
    thu_tu = chon[torch.randperm(len(chon))]
    for i in range(0, len(thu_tu), batch):
        b = thu_tu[i:i + batch]
        mat_mat = F.cross_entropy(m(X[b]), y[b])
        toi_uu.zero_grad()
        mat_mat.backward()
        toi_uu.step()


def do_chinh_xac(m):
    with torch.no_grad():
        return 100 * (m(X_kt).argmax(1) == y_kt).float().mean().item()


dat_hat_giong(0)
tham_do = mlp(32)
toi_uu = torch.optim.Adam(tham_do.parameters(), lr=1e-3, foreach=False)
for _ in range(3):
    mot_epoch(tham_do, toi_uu, torch.arange(N))
with torch.no_grad():
    de_truoc = torch.argsort(F.cross_entropy(tham_do(X), y, reduction="none"))  # mất mát thấp là dễ


def tap_cua_epoch(ke_hoach, n, m):
    k = int(N * min(1.0, n / KHOI_DONG))
    if ke_hoach == "ngẫu nhiên":
        return torch.arange(N)
    if ke_hoach == "lộ trình":
        return de_truoc[:k]
    if ke_hoach == "lộ trình theo lớp":
        return torch.cat([de_truoc[y[de_truoc] == c][:int(k * (y == c).sum() / N)] for c in range(10)])
    if ke_hoach == "lộ trình ngược":
        return de_truoc[-k:]
    with torch.no_grad():  # tự điều nhịp: mất mát của chính mô hình lúc này
        return torch.argsort(F.cross_entropy(m(X), y, reduction="none"))[:k]


KE_HOACH = ["ngẫu nhiên", "lộ trình", "lộ trình theo lớp", "lộ trình ngược", "tự điều nhịp"]
HAT_GIONG = [SEED, SEED + 1, SEED + 2]  # một lần chạy dao động cả điểm phần trăm giữa hai epoch, nên lấy trung bình
duong = {}
for ke_hoach in KE_HOACH:
    cac_lan = []
    for hg in HAT_GIONG:
        dat_hat_giong(hg)
        m = mlp()
        toi_uu = torch.optim.Adam(m.parameters(), lr=1e-3, foreach=False)
        da_xu_ly, diem = 0, []
        for n in range(1, SO_EPOCH + 1):
            chon = tap_cua_epoch(ke_hoach, n, m)
            mot_epoch(m, toi_uu, chon)
            da_xu_ly += len(chon)
            diem.append((da_xu_ly, do_chinh_xac(m)))
        cac_lan.append(diem)
    # số mẫu mỗi epoch không đổi giữa các hạt giống; độ chính xác lấy trung bình ba lần chạy
    duong[ke_hoach] = [(cac_lan[0][e][0], float(np.mean([lan[e][1] for lan in cac_lan]))) for e in range(SO_EPOCH)]

MUC_TIEU = duong["ngẫu nhiên"][-1][1] - 1  # quy tắc của sổ tay: độ chính xác cuối của ngẫu nhiên trừ 1 điểm


def mau_toi_muc_tieu(diem):
    for so_mau, dung in diem:
        if dung >= MUC_TIEU:
            return so_mau
    return None


goc = mau_toi_muc_tieu(duong["ngẫu nhiên"])
for ke_hoach, diem in duong.items():
    can = mau_toi_muc_tieu(diem)
    cuoi = f"cuối {diem[-1][1]:.1f}% sau {diem[-1][0]:,} mẫu"
    if can is None:
        print(f"⏱ {ke_hoach:<18} không đạt {MUC_TIEU:.1f}% ({cuoi}) (đo trên máy này)")
    else:
        print(f"⏱ {ke_hoach:<18} đạt {MUC_TIEU:.1f}% sau {can:>7,} mẫu, tiết kiệm {1 - can / goc:+.0%} so với ngẫu nhiên"
              f" ({cuoi}) (đo trên máy này)")

In [ ]:
MAU = {"ngẫu nhiên": "0.3", "lộ trình": "C2", "lộ trình theo lớp": "C0", "lộ trình ngược": "C3", "tự điều nhịp": "C4"}
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.2))
for ax in (ax1, ax2):
    for ke_hoach, diem in duong.items():
        ax.plot([s / 1000 for s, _ in diem], [d for _, d in diem], "o-", ms=4, lw=1.8, color=MAU[ke_hoach], label=ke_hoach)
    ax.axhline(MUC_TIEU, color="0.5", lw=0.8, ls="--")
    ax.set_xlabel("số mẫu đã xử lý (nghìn), tức tính toán đã trả")
ax1.set_ylabel("độ chính xác trên tập kiểm tra (%)")
ax1.set_title("Cả quá trình (đo trên máy này)")
ax1.legend(loc="lower right", fontsize=8)
ax2.set_xlim(30, 125)
ax2.set_ylim(MUC_TIEU - 4, MUC_TIEU + 2.5)
ax2.text(31, MUC_TIEU + 0.25, f"mục tiêu {MUC_TIEU:.1f}%", fontsize=8, color="0.35")
ax2.set_title("Phóng to quanh mục tiêu (đo trên máy này)")
fig.suptitle("Học theo lộ trình trên FashionMNIST, trung bình 3 hạt giống", fontsize=11)
plt.tight_layout()
plt.show()

Lộ trình dễ trước, xếp hạng trên toàn tập, khởi đầu rất thấp: phần dễ nhất của tập dồn vào vài lớp, nên vài
epoch đầu mô hình gần như không thấy các lớp còn lại. Chia đều theo lớp sửa được phần lớn khởi đầu ấy. Lộ
trình ngược, đưa mẫu khó lên trước, khởi đầu còn thấp hơn nhưng bắt kịp nhanh nhất. Tới gần mục tiêu thì mọi
đường xích lại trong vài điểm phần trăm, và một bước của trục ngang là cả một epoch, nên vài phần trăm
"tiết kiệm" ở bảng trên nằm trong độ thô của chính phép đo.

Đó là đúng điều chương nói: không thứ tự nào luôn hơn, nên mọi mức tăng của lộ trình phải đo ở một chỉ số
mục tiêu cố định, trên đúng tác vụ, như bảng 5 làm với một quy tắc mục tiêu chung. Phần tính toán mà nhịp
độ tuyến tính tiết kiệm trong giai đoạn khởi động dần chỉ là lời nếu độ chính xác theo kịp.

## 2. Học chủ động: hỏi nhãn cho mẫu nào?

Học chủ động giả định nhãn đắt, còn mẫu chưa nhãn thì rẻ. Vòng lặp bắt đầu từ 100 ảnh chữ số có nhãn chọn
ngẫu nhiên, trong một kho 20,000 ảnh. Mỗi vòng, một mô hình logistic chấm mọi ảnh chưa nhãn, chọn 100 ảnh
đáng hỏi nhất, "chuyên gia" trả nhãn (ở đây là nhãn thật của MNIST), rồi mô hình được huấn luyện lại.
Bốn chiến lược truy vấn:

* **độ tự tin thấp nhất**: xác suất của lớp dự đoán nhỏ nhất;
* **biên nhỏ nhất**: hai lớp được dự đoán cao nhất sát nhau nhất;
* **entropy**: phân phối dự đoán phẳng nhất;
* **truy vấn theo hội đồng**: 5 mô hình huấn luyện trên 5 mẫu bootstrap, chọn ảnh chúng bất đồng nhiều nhất.

Ba chiến lược đầu là lấy mẫu theo độ bất định, mỗi cách đo độ bất định một kiểu.

**Dự đoán:** cần bao nhiêu nhãn để đạt độ chính xác mà 1,600 nhãn ngẫu nhiên cho?

In [ ]:
KHO = 20_000
BAT_DAU, BUOC, SO_VONG = 100, 100, 15


def hoi_quy(chi_so):
    return LogisticRegression(C=0.5, max_iter=200).fit(Z[chi_so], nhan[chi_so])


def diem_bat_dinh(chien_luoc, p):
    s = np.sort(p, 1)
    if chien_luoc == "độ tự tin thấp nhất":
        return 1 - s[:, -1]
    if chien_luoc == "biên nhỏ nhất":
        return s[:, -2] - s[:, -1]  # biên càng nhỏ, điểm càng cao
    return -(p * np.log(p + 1e-12)).sum(1)


def hoc_chu_dong(chien_luoc, buoc=BUOC, ngan_sach=BAT_DAU + BUOC * SO_VONG, ghi_duong=True):
    rng = np.random.default_rng(SEED)
    co_nhan = rng.choice(KHO, BAT_DAU, replace=False)  # cùng 100 ảnh khởi đầu cho mọi chiến lược
    duong_, so_lan_cham = [], 0
    while True:
        m = hoi_quy(co_nhan)
        if ghi_duong or len(co_nhan) >= ngan_sach:
            duong_.append((len(co_nhan), 100 * m.score(Z_kt, nhan_kt)))
        if len(co_nhan) >= ngan_sach:
            return duong_, co_nhan, so_lan_cham
        con_lai = np.setdiff1d(np.arange(KHO), co_nhan)
        if chien_luoc == "ngẫu nhiên":
            moi = rng.choice(con_lai, buoc, replace=False)
        elif chien_luoc == "truy vấn theo hội đồng":
            phieu = np.zeros((len(con_lai), 10))
            for _ in range(5):
                thanh_vien = hoi_quy(rng.choice(co_nhan, len(co_nhan), replace=True))
                phieu[np.arange(len(con_lai)), thanh_vien.predict(Z[con_lai])] += 1
            phieu /= 5
            diem_ = -(phieu * np.log(phieu + 1e-12)).sum(1) + 1e-6 * rng.random(len(con_lai))  # phá thế hoà
            moi = con_lai[np.argsort(diem_)[-buoc:]]
            so_lan_cham += 5 * len(con_lai)
        else:
            diem_ = diem_bat_dinh(chien_luoc, m.predict_proba(Z[con_lai]))
            moi = con_lai[np.argsort(diem_)[-buoc:]]
            so_lan_cham += len(con_lai)
        co_nhan = np.concatenate([co_nhan, moi])


CHIEN_LUOC = ["ngẫu nhiên", "độ tự tin thấp nhất", "biên nhỏ nhất", "entropy", "truy vấn theo hội đồng"]
ket_qua, truy_van = {}, {}
for cl in CHIEN_LUOC:
    t0 = time.perf_counter()
    duong_, co_nhan, so_lan_cham = hoc_chu_dong(cl)
    ket_qua[cl], truy_van[cl] = duong_, co_nhan[BAT_DAU:]
    print(f"⏱ {cl:<23} {duong_[-1][1]:.1f}% với {duong_[-1][0]:,} nhãn; chấm {so_lan_cham:>9,} lượt dự đoán,"
          f" {time.perf_counter() - t0:.1f} s (đo trên máy này)")

MUC_TIEU_AL = ket_qua["ngẫu nhiên"][-1][1]
print(f"\nmục tiêu: {MUC_TIEU_AL:.1f}%, độ chính xác của {ket_qua['ngẫu nhiên'][-1][0]:,} nhãn ngẫu nhiên")


def nhan_toi_muc_tieu(duong_):
    return next((n for n, d in duong_ if d >= MUC_TIEU_AL), None)


goc_al = nhan_toi_muc_tieu(ket_qua["ngẫu nhiên"])
for cl, duong_ in ket_qua.items():
    can = nhan_toi_muc_tieu(duong_)
    if can is None:
        print(f"{cl}: chưa đạt mục tiêu trong ngân sách")
        continue
    print(f"⏱ {cl:<23} cần {can:>5,} nhãn, hiệu quả nhãn {goc_al / can:.1f}× so với ngẫu nhiên (đo trên máy này)")

In [ ]:
MAU_AL = {"ngẫu nhiên": "0.3", "độ tự tin thấp nhất": "C1", "biên nhỏ nhất": "C3", "entropy": "C0",
          "truy vấn theo hội đồng": "C4"}
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.2), gridspec_kw={"width_ratios": [1.15, 1]})
for cl, duong_ in ket_qua.items():
    ax1.semilogx([n for n, _ in duong_], [d for _, d in duong_], "o-", ms=3, lw=1.8, color=MAU_AL[cl], label=cl)
ax1.axhline(MUC_TIEU_AL, color="0.5", lw=0.8, ls="--")
ax1.set_xticks([100, 200, 400, 800, 1600], ["100", "200", "400", "800", "1,600"])
ax1.minorticks_off()
ax1.set_xlabel("số nhãn đã mua (thang log)")
ax1.set_ylabel("độ chính xác trên tập kiểm tra (%)")
ax1.set_title("Đường cong học tập của học chủ động (đo trên máy này)")
ax1.legend(loc="lower right", fontsize=8)

trong_kho = np.bincount(nhan[:KHO], minlength=10) / KHO
for j, cl in enumerate(("ngẫu nhiên", "biên nhỏ nhất")):
    ti_le = np.bincount(nhan[truy_van[cl]], minlength=10) / len(truy_van[cl])
    ax2.bar(np.arange(10) + (j - 0.5) * 0.4, 100 * ti_le, width=0.4, color=MAU_AL[cl], label=f"truy vấn: {cl}")
ax2.plot(range(10), 100 * trong_kho, "k_", ms=18, mew=2, label="tỉ lệ trong kho")
ax2.set_xticks(range(10))
ax2.set_xlabel("chữ số")
ax2.set_ylabel("phần trăm số truy vấn")
ax2.set_ylim(0, 1.45 * ax2.get_ylim()[1])
ax2.set_title("Truy vấn dồn vào đâu (đo trên máy này)")
ax2.legend(loc="upper center", fontsize=8, ncol=2)
plt.tight_layout()
plt.show()

Không phải chiến lược độ bất định nào cũng thắng như nhau: ở lần chạy này biên nhỏ nhất đạt mục tiêu với
số nhãn ít hơn hẳn, độ tự tin thấp nhất chỉ hơn ngẫu nhiên chút ít, còn entropy kết thúc dưới cả ngẫu
nhiên. Biểu đồ cột cho thấy truy vấn đi đâu: biên nhỏ nhất nghiêng hẳn về vài chữ số dễ nhầm với nhau và
bỏ bớt những chữ số mô hình đã chắc. Đó vừa là lý do nó hiệu quả, vừa là cái giá chương nêu: nó "có thể
dồn quá nhiều truy vấn vào cùng một vùng mơ hồ".

Hình 6 của chương vẽ khoảng cách chừng 4 lần ở gần 90 phần trăm, và chương ghi đó là kịch bản minh hoạ.
Khoảng cách đo được tuỳ mô hình, dữ liệu và cỡ lô. Cột "lượt dự đoán" ở trên là cái giá còn lại: mỗi vòng
phải chấm cả kho, và hội đồng 5 mô hình tốn gấp 5 lần. Vì vậy chương dặn phần tiết kiệm trọn vẹn có thể nhỏ
hơn khoảng cách số nhãn.

## 3. Bệnh viện, và độ trễ gán nhãn

Phép tính nhẩm 1.3: một bệnh viện có 1 triệu ảnh chưa nhãn, nhãn chuyên gia 5 đô la mỗi nhãn, ngân sách
500,000 đô la và 1 tháng.

In [ ]:
ANH_BV = sach(1e6, nguon="1 triệu · 5 đô la · 500,000 đô la · 1 tháng")
GIA_NHAN = sach(5, nguon="1 triệu · 5 đô la")
NGAN_SACH = sach(500_000, nguon="5 đô la · 500,000 đô la")
theo_sach("gán nhãn toàn bộ (đô la)", ANH_BV * GIA_NHAN, sach=5_000_000, nguon="5,000,000 đô la (10×) · 100,000")
theo_sach("so với ngân sách", ANH_BV * GIA_NHAN / NGAN_SACH, sach=10, nguon="5,000,000 đô la (10×)")
moc = NGAN_SACH / GIA_NHAN
theo_sach("mốc ngẫu nhiên cùng ngân sách (ảnh)", moc, sach=100_000, nguon="5,000,000 đô la (10×) · 100,000")
CHON_AL = sach(50_000, nguon="50,000 · 250,000 đô la · 50 phần trăm · 2×")
theo_sach("học chủ động: chi phí (đô la)", CHON_AL * GIA_NHAN, sach=250_000, nguon="50,000 · 250,000 đô la")
theo_sach("thấp hơn ngân sách (%)", 100 * (1 - CHON_AL * GIA_NHAN / NGAN_SACH), sach=50,
          nguon="250,000 đô la · 50 phần trăm")
theo_sach("số mẫu mỗi epoch ít hơn", moc / CHON_AL, sach=2, nguon="50 phần trăm · 2×")
theo_sach("tiết kiệm so với mốc tiêu hết ngân sách (đô la)", NGAN_SACH - CHON_AL * GIA_NHAN, sach=250_000,
          nguon="50,000 · 250,000 đô la")
print("độ chính xác của 50,000 ảnh chọn lọc so với 100,000 ảnh ngẫu nhiên vẫn phải kiểm chứng bằng thực nghiệm")

Phép tính chỉ nói về tiền nhãn. Nó không nói 50,000 ảnh chọn theo độ bất định có bằng 100,000 ảnh ngẫu
nhiên không: chương ghi rõ điều đó cần kiểm chứng bằng thực nghiệm, và phần 2 vừa cho thấy câu trả lời
tuỳ chiến lược.

Cạm bẫy thứ tư của chương là bỏ qua **độ trễ gán nhãn**. Chương lấy 14 ngày cho một lô làm ví dụ minh hoạ.
Lô càng nhỏ thì mô hình càng được cập nhật thường xuyên, nhưng số vòng, và số lần chờ, càng nhiều. Chạy lại
vòng biên nhỏ nhất với cùng 1,000 nhãn mua thêm, chỉ đổi cỡ lô.

**Dự đoán:** lô 10 nhãn và lô 1,000 nhãn chênh nhau bao nhiêu điểm, và bao nhiêu ngày?

In [ ]:
NGAY_MOI_LO = sach(14, nguon="14 ngày")
CO_LO = [10, 50, 100, 250, 500, 1_000]
theo_lo = []
for lo in CO_LO:
    duong_, _, _ = hoc_chu_dong("biên nhỏ nhất", buoc=lo, ngan_sach=BAT_DAU + 1_000, ghi_duong=False)
    so_lo = 1_000 // lo
    theo_lo.append((lo, so_lo, so_lo * NGAY_MOI_LO, duong_[-1][1]))
    print(f"⏱ lô {lo:>5,} nhãn: {so_lo:>3} vòng, {so_lo * NGAY_MOI_LO:>5,} ngày chờ nhãn, độ chính xác {duong_[-1][1]:.1f}%"
          " (đo trên máy này)")

fig, ax = plt.subplots(figsize=(7.5, 3.6))
ax.semilogx([n for _, _, n, _ in theo_lo], [d for *_, d in theo_lo], "o-", color="C3", lw=2)
for lo, _, ngay, d in theo_lo:
    ax.annotate(f"lô {lo:,}", (ngay, d), textcoords="offset points", xytext=(0, 7), ha="center", fontsize=8)
ngau_nhien_1100 = next(d for n, d in ket_qua["ngẫu nhiên"] if n == BAT_DAU + 1_000)
ax.axhline(ngau_nhien_1100, color="0.4", lw=1, ls="--")
ax.text(ax.get_xlim()[1] * 0.9, ngau_nhien_1100 - 0.45, "ngẫu nhiên, cùng 1,100 nhãn", fontsize=8, ha="right", color="0.3")
ax.set_xlabel(f"ngày chờ nhãn, {NGAY_MOI_LO} ngày mỗi lô (thang log)")
ax.set_ylabel("độ chính xác sau 1,100 nhãn (%)")
ax.set_ylim(min(d for *_, d in theo_lo) - 2.5, max(d for *_, d in theo_lo) + 0.8)
ax.set_title("Cỡ lô truy vấn: thích ứng hay thời gian (đo trên máy này)")
plt.tight_layout()
plt.show()

Từ lô 50 tới lô 500, độ chính xác gần như không đổi, trong khi thời gian chờ chênh nhau mười lần. Chỉ ở hai
đầu mới có cái giá thật: một lô 1,000 duy nhất rơi về gần đường ngẫu nhiên, vì mọi truy vấn được chọn bởi
một mô hình chỉ biết 100 nhãn, còn lô 10 thì đòi 100 lần chờ mà không chính xác hơn. Chương vì vậy nói cỡ lô đúng tuỳ
năng lực gán nhãn đo được, nhịp cập nhật mô hình và yêu cầu đa dạng, và tỉ suất hoàn vốn (ROI) của học
chủ động phụ thuộc cả giá nhãn lẫn thời gian chờ.

## 4. Học bán giám sát: 500 nhãn trên 50,000 ảnh

Kịch bản của chương có 50,000 ảnh X-quang ngực, trong đó 500 ảnh có nhãn. Sổ tay dựng đúng tỉ lệ ấy trên
50,000 ảnh chữ số MNIST. Cách đầu tiên là **gán nhãn giả** (pseudo-labeling): huấn luyện trên 500 nhãn,
gán nhãn giả cho những dự đoán đủ tự tin, rồi huấn luyện lại trên cả hai.

Chương cảnh báo học bán giám sát gặp khó khi kho chưa nhãn chứa mẫu ngoài phân phối. Để thấy điều đó, đặt
thêm 10,000 ảnh FashionMNIST vào kho: không ảnh nào là chữ số.

**Dự đoán:** với ngưỡng tự tin 0.95, bao nhiêu ảnh quần áo sẽ được gán nhãn là một chữ số?

In [ ]:
TONG_SSL = sach(50_000, nguon="50,000 · 500 · 1 phần trăm · 49,500")
CO_NHAN = sach(500, nguon="50,000 · 500")
theo_sach("phần có nhãn (%)", 100 * CO_NHAN / TONG_SSL, sach=1, nguon="50,000 · 500 · 1 phần trăm")
theo_sach("ảnh chưa nhãn", TONG_SSL - CO_NHAN, sach=49_500, nguon="1 phần trăm · 49,500")

rng = np.random.default_rng(SEED)
co_nhan = rng.choice(TONG_SSL, CO_NHAN, replace=False)
chua_nhan = np.setdiff1d(np.arange(TONG_SSL), co_nhan)
goc_ssl = hoi_quy(co_nhan)
du_toan = hoi_quy(np.arange(TONG_SSL))
do_tren_may(f"chỉ {CO_NHAN} nhãn", 100 * goc_ssl.score(Z_kt, nhan_kt), "%")
do_tren_may(f"cả {TONG_SSL:,} nhãn, mức trần của mô hình này", 100 * du_toan.score(Z_kt, nhan_kt), "%")

Z_ngoai = dac_trung(thoi_trang.data[N:N + 10_000].reshape(-1, 784).numpy().astype(np.float32) / 255)
p = goc_ssl.predict_proba(Z[chua_nhan])
tu_tin, nhan_gia = p.max(1), p.argmax(1)
p_ngoai = goc_ssl.predict_proba(Z_ngoai)
tu_tin_ngoai, nhan_gia_ngoai = p_ngoai.max(1), p_ngoai.argmax(1)

NGUONG_FIXMATCH = sach(0.95, nguon="250 (25 mỗi lớp) · 50 nghìn · 0.95")
NGUONG = [0.5, 0.8, 0.9, NGUONG_FIXMATCH, 0.99]
print(f"\n{'ngưỡng':>6} {'nhãn giả nhận':>14} {'đúng':>7} {'độ chính xác sau khi học lại':>29} {'ảnh quần áo lọt vào':>20}")
bang_gia = []
for nguong in NGUONG:
    nhan_vao = tu_tin >= nguong
    m = LogisticRegression(C=0.5, max_iter=200).fit(
        np.concatenate([Z[co_nhan], Z[chua_nhan][nhan_vao]]), np.concatenate([nhan[co_nhan], nhan_gia[nhan_vao]]))
    dung = (nhan_gia[nhan_vao] == nhan[chua_nhan][nhan_vao]).mean()
    so_lot = int((tu_tin_ngoai >= nguong).sum())
    bang_gia.append((nguong, nhan_vao.mean(), dung, (tu_tin_ngoai >= nguong).mean()))
    print(f"{nguong:>6} {int(nhan_vao.sum()):>14,} {dung:>7.1%} {100 * m.score(Z_kt, nhan_kt):>28.1f}% {so_lot:>14,} / 10,000")

lot = tu_tin_ngoai >= NGUONG_FIXMATCH
m_lan = LogisticRegression(C=0.5, max_iter=200).fit(
    np.concatenate([Z[co_nhan], Z[chua_nhan][tu_tin >= NGUONG_FIXMATCH], Z_ngoai[lot]]),
    np.concatenate([nhan[co_nhan], nhan_gia[tu_tin >= NGUONG_FIXMATCH], nhan_gia_ngoai[lot]]))
do_tren_may(f"ngưỡng {NGUONG_FIXMATCH}, kho có lẫn ảnh quần áo", 100 * m_lan.score(Z_kt, nhan_kt), "%")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(NGUONG, [100 * b[1] for b in bang_gia], "o-", color="C0", lw=2, label="chữ số được nhận nhãn giả")
ax1.plot(NGUONG, [100 * b[2] for b in bang_gia], "s--", color="C2", lw=1.5, label="nhãn giả đúng")
ax1.plot(NGUONG, [100 * b[3] for b in bang_gia], "^-", color="C3", lw=2, label="ảnh quần áo được nhận")
ax1.axvline(NGUONG_FIXMATCH, color="0.5", lw=0.8, ls=":")
ax1.text(NGUONG_FIXMATCH - 0.005, 3, "ngưỡng của FixMatch", fontsize=8, ha="right", color="0.35")
ax1.set_xlabel("ngưỡng tự tin")
ax1.set_ylabel("phần trăm")
ax1.set_ylim(0, 135)
ax1.set_title("Ngưỡng cao: ít nhãn giả hơn, đúng hơn (đo trên máy này)")
ax1.legend(loc="upper left", fontsize=8, ncol=2)

ax2.bar(range(10), np.bincount(nhan_gia_ngoai[lot], minlength=10), color="C3")
ax2.set_xticks(range(10))
ax2.set_xlabel("chữ số được gán cho ảnh quần áo")
ax2.set_ylabel("số ảnh")
ax2.set_title(f"{int(lot.sum()):,} ảnh quần áo tự tin ≥ {NGUONG_FIXMATCH} (đo trên máy này)")
plt.tight_layout()
plt.show()

Ngưỡng là then chốt, đúng như chương nói: thấp quá thì đưa nhiễu nhãn vào, cao quá thì bỏ phí dữ liệu. Ở
đây gán nhãn giả chỉ thêm được chút ít so với 500 nhãn, còn xa mức trần của mô hình. Lý do là mô hình chỉ
học lại chính những gì nó đã tự tin: phần nó còn sai, nó không bao giờ tự sửa được.

Còn kho có lẫn ảnh quần áo thì một mô hình chưa từng thấy quần áo vẫn gán nhãn chữ số cho hàng nghìn ảnh ở
ngưỡng 0.8, hơn một nghìn ảnh ngay cả ở ngưỡng 0.95, và dồn gần hết vào một hai chữ số. Đó chính là hai chế
độ hỏng chương liệt kê: mẫu ngoài phân phối bị gán sai nhãn một cách tự tin, và nhãn giả dồn lệch về vài
lớp. Tập kiểm tra ở đây chỉ có chữ số nên độ chính xác gần như không đổi; cái hỏng là mô hình giờ được dạy
rằng một chiếc áo là một chữ số. Độ tự tin của một mô hình không phải là bằng chứng mẫu thuộc về phân phối.

### Lan truyền nhãn

Lan truyền nhãn (label propagation) dựng một đồ thị tương đồng trên mọi mẫu rồi truyền nhãn sang hàng xóm.
Chương nói nó hợp nhất khi không gian đặc trưng có cụm rõ ràng, và chữ số MNIST là một trường hợp như vậy.
Đồ thị 10 hàng xóm gần nhất trên 20,000 ảnh tốn bộ nhớ và thời gian, nên lần này dùng 20,000 ảnh với 200
nhãn, giữ tỉ lệ 1 phần trăm.

In [ ]:
SO_ANH_LT = 20_000
nhan_lt = round(SO_ANH_LT * CO_NHAN / TONG_SSL)
chon_lt = np.random.default_rng(SEED).choice(SO_ANH_LT, nhan_lt, replace=False)
y_lt = np.full(SO_ANH_LT, -1)
y_lt[chon_lt] = nhan[chon_lt]
t0 = time.perf_counter()
lan = LabelSpreading(kernel="knn", n_neighbors=10, max_iter=100).fit(Z[:SO_ANH_LT], y_lt)
do_tren_may("lan truyền nhãn: thời gian", time.perf_counter() - t0, "s")
do_tren_may(f"logistic, chỉ {nhan_lt} nhãn", 100 * hoi_quy(chon_lt).score(Z_kt, nhan_kt), "%")
do_tren_may("nhãn lan truyền đúng trên 20,000 ảnh", 100 * (lan.transduction_ == nhan[:SO_ANH_LT]).mean(), "%")
do_tren_may("lan truyền nhãn, trên tập kiểm tra", 100 * lan.score(Z_kt, nhan_kt), "%")

Cùng 1 phần trăm nhãn, lan truyền nhãn bỏ xa mô hình logistic chỉ học từ phần có nhãn, vì nó dùng chính cấu
trúc của dữ liệu chưa nhãn: giả định cụm, biên nên nằm ở vùng thưa. Trên một tập mà các lớp không tách
thành cụm, cũng thuật toán ấy có thể truyền nhãn sai đi xa. Chương vì vậy dặn luôn kiểm định trên một tập
giữ riêng có nhãn thật: không kỹ thuật nào ở đây bỏ được nhu cầu đánh giá có nhãn.

## 5. FixMatch trên CIFAR-10

FixMatch kết hợp gán nhãn giả với điều chuẩn nhất quán (consistency regularization): chỉ gán nhãn giả khi
dự đoán trên ảnh tăng cường yếu đủ tự tin, trên 0.95, rồi dạy mô hình đoán nhãn ấy trên bản tăng cường
mạnh. Bảng 6 so nó với tập huấn luyện đầy đủ 50,000 nhãn. Huấn luyện FixMatch cần GPU và hàng trăm epoch,
nên ở đây chỉ kiểm phép tính.

In [ ]:
DAY_DU_B6 = sach(50_000, nguon="250 (25 mỗi lớp) · 50 nghìn")
DONG_B6 = [  # số nhãn, độ chính xác của FixMatch
    (sach(4_000, nguon="4,000 (8 phần trăm) 95.7"), sach(95.7, nguon="4,000 (8 phần trăm) 95.7")),
    (sach(250, nguon="250 (0.5 phần trăm) 94.9"), sach(94.9, nguon="250 (0.5 phần trăm) 94.9")),
    (sach(40, nguon="40 (0.08 phần trăm) 88.6"), sach(88.6, nguon="40 (0.08 phần trăm) 88.6")),
]
theo_sach("4,000 nhãn: phần của tập (%)", 100 * DONG_B6[0][0] / DAY_DU_B6, sach=8, nguon="4,000 (8 phần trăm)")
theo_sach("250 nhãn: phần của tập (%)", 100 * DONG_B6[1][0] / DAY_DU_B6, sach=0.5, nguon="250 (0.5 phần trăm)")
theo_sach("40 nhãn: phần của tập (%)", 100 * DONG_B6[2][0] / DAY_DU_B6, sach=0.08, nguon="40 (0.08 phần trăm)")
theo_sach("4,000 nhãn: ít nhãn hơn", DAY_DU_B6 / DONG_B6[0][0], sach=12.5, nguon="12.5× · 200× · 1250×")
theo_sach("250 nhãn: ít nhãn hơn", DAY_DU_B6 / DONG_B6[1][0], sach=200, nguon="12.5× · 200× · 1250×")
theo_sach("40 nhãn: ít nhãn hơn", DAY_DU_B6 / DONG_B6[2][0], sach=1250, nguon="12.5× · 200× · 1250×")
for so_nhan, dung in DONG_B6:
    print(f"   {so_nhan:>5,} nhãn: FixMatch đạt {dung}%")
NHAN_250 = sach(250, nguon="250 (25 mỗi lớp)")
theo_sach("nhãn mỗi lớp ở dòng 250 nhãn", NHAN_250 / 10, sach=25, nguon="250 (25 mỗi lớp)")

GIAM_CHI_PHI = sach(8.1, nguon="0.95 · 94.9 phần trăm · 8.1×")
print(f"\nchương: chi phí chuẩn bị dữ liệu giảm {GIAM_CHI_PHI}×, trong khi số nhãn giảm {DAY_DU_B6 / NHAN_250:.0f}×")
# Một giả định khớp được con số, KHÔNG phải giả định của chương: chi phí = nhãn · c + phần cố định F.
F_theo_nhan = (DAY_DU_B6 - GIAM_CHI_PHI * NHAN_250) / (GIAM_CHI_PHI - 1)
print(f"nếu có một phần chi phí cố định không phụ thuộc số nhãn, nó phải bằng giá của khoảng {F_theo_nhan:,.0f} nhãn")

Bốn tỉ lệ của bảng 6 khép kín. Con số 8.1 lần thì không suy ra được từ những gì đoạn trích ghi: chương nói nó
đến từ "giả định chi phí nhãn của chương" mà không nêu giả định ấy. Dòng cuối chỉ là một cách đọc: nếu
chi phí chuẩn bị gồm tiền nhãn cộng một phần cố định, phần cố định ấy phải cỡ giá của gần 7,000 nhãn thì
200 lần ít nhãn mới chỉ còn 8.1 lần rẻ hơn. Điều chắc chắn là chương nói thêm: phần tiết kiệm trọn vẹn còn
tuỳ chi phí dữ liệu chưa nhãn và tính toán, vì học bán giám sát đổi tính toán huấn luyện lấy nhãn.

## Thử thêm

1. Ở phần 1, thêm nhiễu nhãn 40 phần trăm vào `y` trước khi chạy (xem sổ tay 01). Dòng cuối của bảng 5 nói
   về lộ trình trên nhãn nhiễu; ở đây lộ trình có đổi được thứ hạng với xáo trộn ngẫu nhiên không?
2. Ở phần 2, thêm chiến lược đa dạng: chọn 100 ảnh xa nhất với tập đã có nhãn trong không gian `Z`, bằng
   cùng thuật toán k-Center của sổ tay 02. Nó đứng ở đâu so với biên nhỏ nhất, và truy vấn có đều lớp hơn?
3. Ở phần 4, đổi `max_iter=200` thành `1000` trong `hoi_quy`. Mức trần của mô hình đổi bao nhiêu, và kết luận
   về gán nhãn giả có đổi không?

## Tóm lại

* Học theo lộ trình tiết kiệm tính toán trong giai đoạn khởi động dần, nhưng thứ tự dễ trước có thể bỏ đói
  cả lớp lúc đầu, và gần mục tiêu các thứ tự chỉ cách nhau trong độ thô của phép đo; mức tăng phải đo ở một
  mục tiêu cố định.
* Học chủ động mua được cùng độ chính xác với ít nhãn hơn, nhưng chiến lược quyết định nhiều, truy vấn dồn
  vào vùng mơ hồ, và mỗi vòng phải chấm cả kho.
* Cỡ lô truy vấn đổi độ thích ứng lấy thời gian chờ nhãn; ROI của học chủ động phụ thuộc cả hai.
* Gán nhãn giả chỉ học lại điều mô hình đã tự tin, và tin cả mẫu ngoài phân phối; lan truyền nhãn khai thác
  được cấu trúc cụm khi cấu trúc ấy có thật.
* Bảng 5, phép tính nhẩm 1.3 và bảng 6 khép kín; con số 8.1 lần của FixMatch dựa trên giả định mà chương không
  ghi ra.

---
### Nguồn và giấy phép

Sổ tay này đi kèm bài học [Chọn lọc dữ liệu — bớt việc trước khi việc bắt đầu](https://d3lu8vk4we0dc.cloudfront.net/lessons/vol1-ch09-data-selection/), dựng từ chương
[*Data Selection*](https://mlsysbook.ai/vol1/data_selection/data_selection.html) của sách.

Nội dung gốc thuộc Machine Learning Systems của Vijay Janapa Reddi và cộng sự, MIT Press. Bản quyền © 2024-2026 Harvard University. Giấy phép CC BY-NC-SA 4.0. Bản tiếng Việt này là tác phẩm phái sinh dùng cùng giấy phép: ghi công, chia sẻ tương tự, phi thương mại.